# EmpowerLens — DeBERTa-v3 vs MentalRoBERTa, combined-data Kaggle GPU runner

Fine-tunes **microsoft/deberta-v3-base** and **mental/mental-roberta-base** back-to-back on the **combined** training split (Annotated_data.csv train + CODIPAS message-level train, concatenated — see `src/make_splits_combined.py`), reusing `train_transformer.py` / `evaluate.py` completely unmodified. Evaluation stays on the frozen Annotated_data.csv val/test, so results are directly comparable to your existing `roberta-base` / `mental-bert-base-uncased` rows in `results/paper_comparison.csv`.

**Before running:**
1. Settings -> **Accelerator: GPU**, **Internet: On**.
2. Locally, run once and commit/push:
   ```
   python -m src.make_splits_codipas_classification --force   # if not already done
   python -m src.make_splits_combined --force
   ```
   `data/splits_combined/{train,val,test}.csv` + `split_manifest.json` must already be on the branch below — this notebook uses them as-is.
3. `HF_TOKEN` secret should be set (Add-ons -> Secrets) in case either model repo is gated on Hugging Face — verify gating/repo IDs on HF before a long run; `mental/mental-roberta-base` may or may not require the same acceptance step as `mental-bert-base-uncased` did.
4. Two models x 3 seeds x however many TASKS you select below = real GPU time. Default is `multiclass` only (your weakest task); uncomment `ALL_TASKS` to sweep all three.

In [ ]:
# 1. Clone the repo and install the transformer stack.
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt
!pip install -q sentencepiece  # deberta-v3 tokenizer needs this

In [ ]:
# 1b. Log in in case mental-roberta (or any future swap) is gated like mental-bert was.
#     Harmless no-op if neither model needs it.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
except Exception as e:
    print(f"[warn] no HF_TOKEN secret found or login failed ({e}); "
          f"continuing — only fails downstream if a model repo turns out to be gated.")

In [ ]:
# 1c. Build the combined split ON KAGGLE if you didn't commit data/splits_combined/.
#     Safe to skip (or leave commented) if it's already on the branch — the script
#     refuses to overwrite existing files without --force anyway.
import os
if not os.path.exists("data/splits_combined/train.csv"):
    !python -m src.make_splits_combined --force

In [ ]:
# 2. Choose task(s), train BOTH models over 3 seeds each on the combined split,
#    evaluate each on val+test. --device auto resolves to the Kaggle GPU.
TASKS = ["multiclass"]              # default: your weakest task
# TASKS = ["binary", "multiclass", "multilabel"]   # uncomment for the full sweep

MODELS = [
    "microsoft/deberta-v3-base",
    "mental/mental-roberta-base",
]

SPLITS = "data/splits_combined"
OUT_CKPT = "checkpoints_combined"
OUT_RES = "results_combined"

for MODEL in MODELS:
    for TASK in TASKS:
        for seed in (42, 1337, 2024):
            ckpt = f"{OUT_CKPT}/{TASK}_{MODEL.split('/')[-1]}_{seed}"
            !python -m src.train_transformer --task $TASK --model $MODEL --seed $seed \
                --device auto --splits $SPLITS --out $OUT_CKPT
            !python -m src.evaluate --checkpoint $ckpt --splits $SPLITS --out $OUT_RES --reference

In [ ]:
# 3. Aggregate, then head-to-head compare the two new models against each other.
#    To compare against your existing roberta-base / mental-bert-base-uncased runs
#    too, copy those eval_*.json files from results/ into results_combined/ first.
!python -m src.aggregate --results $OUT_RES
!python -m src.compare_models --results $OUT_RES --models microsoft/deberta-v3-base,mental/mental-roberta-base --split test

In [ ]:
# 4. Copy results to Kaggle output, prune the heavy stuff so Output stays downloadable.
!mkdir -p /kaggle/working/results_combined
!cp -r $OUT_RES/* /kaggle/working/results_combined/
!rm -rf /kaggle/working/empowerlens/$OUT_CKPT
!rm -rf /kaggle/working/empowerlens/.git
!ls -la /kaggle/working/results_combined